# NovaMart RAG Pipeline — Validation Report

This notebook documents the build, testing, and validation of a RAG (Retrieval-Augmented Generation) pipeline built for **NovaMart**, a fictional 25-store restaurant chain, used as a demo for a healthcare-adjacent SaaS background but applied here to retail operations data.

**What this notebook covers:**
1. Pipeline overview — what was built and how
2. Bug #1 — broad/comparative questions returning incomplete answers
3. The fix — classifying questions and routing to structured retrieval
4. Bug #2 — a second retrieval gap found during re-validation
5. Full validation — every claim checked against raw source data, for 5 test questions across all 4 data types
6. Takeaways

**Why this matters:** RAG systems can produce fluent, confident-sounding answers that are subtly or significantly wrong if retrieval doesn't surface the right data. This notebook shows the actual process of catching that — not just the polished end result.


## 1. Setup

Load the synthetic data used by the pipeline. This is the same data the RAG system was built on top of — five CSVs covering store metadata, sales, inventory, labor, and customer satisfaction across 25 stores and 12 weeks.


In [ ]:
import pandas as pd

# Load source data — same files used to build the RAG pipeline's vector store
stores   = pd.read_csv('data/store_master.csv')
sales    = pd.read_csv('data/weekly_sales.csv')
inventory = pd.read_csv('data/inventory.csv')
labor    = pd.read_csv('data/labor.csv')
customer = pd.read_csv('data/customer.csv')

print(f"Stores:    {len(stores)} rows")
print(f"Sales:     {len(sales)} rows")
print(f"Inventory: {len(inventory)} rows")
print(f"Labor:     {len(labor)} rows")
print(f"Customer:  {len(customer)} rows")


## 2. Pipeline Overview

The pipeline works in three stages:

1. **Ingestion** (`ingest.py`) — each CSV row is converted into a natural-language sentence (e.g. *"NovaMart Miami sales for 2024-W12: Revenue was $72,068 against a target of $100,250, 28.1% below target..."*), embedded with OpenAI's `text-embedding-3-small`, and stored in ChromaDB with metadata (`type`, `store_id`, `week`).

2. **Retrieval** (`query.py`) — when a question comes in, it's classified as either:
   - **Narrow** (about one specific store/week) → semantic vector search, top 8 chunks
   - **Broad** (comparative, spans many stores) → structured retrieval that pulls the most recent week for *every* store directly from ChromaDB's metadata, bypassing similarity search entirely

3. **Generation** — retrieved chunks are passed to Claude along with the question, and Claude generates the final answer.

4. **Interface** (`slack_bot.py`) — a Slack bot using `slack_bolt`'s lazy-listener pattern, so slow RAG/Claude calls never block Slack's connection acknowledgment.

The rest of this notebook focuses on stage 2 — retrieval — because that's where both bugs were found.


## 3. Bug #1 — Broad Questions Returned Incomplete Answers

**The setup:** the original retrieval logic embedded the question and pulled the top 8 most semantically similar chunks out of a corpus of 1,000+ chunks (25 stores × 12 weeks × 4 data types).

**The question asked:** *"Which stores are underperforming against revenue targets?"*

**What came back:** only 3 stores were cited — New York, Atlanta, and Minneapolis. Two of the three had factual errors:

- **Atlanta** — the bot claimed *"no W11 data yet"* and treated it as the most urgent unresolved case. The W11 data existed in the vector store; it simply didn't score high enough on embedding similarity to make the top 8.
- **New York** — the bot claimed a conversion rate was *"the lowest in recent history."* Three earlier weeks were actually lower — they just weren't retrieved either.

**Why this happened:** pure semantic similarity has no concept of "give me one row per store" or "give me the most recent week for everyone." For a broad, comparative question, it just returns whichever 8 chunks happen to score highest — which skews toward whichever rows have the most extreme language, not balanced coverage across the dataset.

The model wasn't hallucinating. It was reasoning correctly over an incomplete, semantically-biased sample — and the gap wasn't visible without checking the raw source data directly.


In [ ]:
# Reproducing the Atlanta (S021) data gap that the bot incorrectly claimed didn't exist

atlanta = sales[(sales['store_id'] == 'S021')].sort_values('week')
print("Atlanta (S021) — full sales history:")
print(atlanta[['week', 'revenue_actual', 'revenue_target', 'revenue_variance_pct', 'same_store_sales_growth']].to_string(index=False))


**Confirmed:** W11 data for Atlanta exists in the source data (and was confirmed to exist in the vector store's index as well). The bot's claim of "no data yet" was false — a retrieval gap, not a data gap.


## 4. The Fix — Question Classification + Structured Retrieval

The fix has two parts, both in `query.py`:

**1. `is_broad_question()`** — a fast, keyword-based check (no extra API call) that flags comparative questions using patterns like `which stores`, `where are`, `compare`, `worst`, `underperform`.

**2. `retrieve_chunks_structured()`** — for broad questions, instead of trusting vector similarity, this loops through all 25 store IDs and pulls each store's most recent week directly via ChromaDB's metadata filter (`where={"type": ..., "store_id": ...}`). This guarantees every store is represented — nothing can be crowded out by unrelated chunks scoring higher on similarity.

Narrow questions (e.g. *"how did S016 do last week"*) still use the original semantic path, since that's the right tool for a specific, single-entity lookup.


In [ ]:
import re

# Simplified version of the classifier from query.py, for demonstration

BROAD_PATTERNS = [
    r"\bwhich stores?\b", r"\bwhat stores?\b", r"\bwhere (are|is)\b",
    r"\bacross (all )?(stores?|locations?)\b", r"\bcompare\b", r"\bunderperform",
    r"\boutperform", r"\bworst\b", r"\bbest\b",
]

def is_broad_question(question: str) -> bool:
    q = question.lower()
    return any(re.search(p, q) for p in BROAD_PATTERNS)

test_questions = [
    "Which stores are underperforming against revenue targets?",
    "How did store S016 do last week?",
    "Where are labor costs running over budget?",
]

for q in test_questions:
    print(f"{'BROAD' if is_broad_question(q) else 'NARROW':8} | {q}")


**After the fix**, re-asking the same revenue question returned **10 stores**, fully ranked, with every figure checked against source data (see Section 6 for the full validation).


## 5. Bug #2 — A Second Gap, Found During Re-Validation

After the fix, a new question was tested: *"Where are labor costs running over budget?"*

**What came back:** the same failure mode as before — only 2 of 25 stores cited (Chicago and LA), stale week ranges, and a confident "data gap" disclaimer that wasn't true.

**Root cause:** the broad/narrow classifier worked correctly, but this specific phrasing — *"where are X running over budget"* — didn't match any of the original keyword patterns (`which stores`, `underperform`, etc.). It fell through to the narrow/semantic path by default, inheriting the exact same retrieval-coverage problem as before.


In [ ]:
# Demonstrating the gap: this phrasing fails the ORIGINAL keyword list

ORIGINAL_BROAD_PATTERNS = [
    r"\bwhich stores?\b", r"\bacross (all )?(stores?|locations?)\b", r"\bcompare\b",
    r"\bunderperform", r"\boutperform", r"\bworst\b", r"\bbest\b",
]

def is_broad_original(question: str) -> bool:
    q = question.lower()
    return any(re.search(p, q) for p in ORIGINAL_BROAD_PATTERNS)

q = "Where are labor costs running over budget?"
print(f"Classified as: {'BROAD' if is_broad_original(q) else 'NARROW'}")
print("This question should be BROAD — it's asking across the whole network.")
print("The original pattern list had no 'where are' pattern, so it fell through.")


**The fix:** expanded the keyword list to include `where (are|is)`, `what stores?`, `list (the )?stores?`, and a few other common phrasings.

**The honest limitation:** keyword-based classification will always have gaps. There's an unlimited number of ways to phrase "show me all stores struggling with X." Expanding the list closed *this* gap, but it's still pattern-matching surface language, not understanding intent. A more robust version would use a cheap LLM call to classify intent — costing a few extra milliseconds and fractions of a cent per query, but generalizing far better than any keyword list. That's a reasonable next step for a production system; a known, documented limitation for a demo.


## 6. Full Validation — Every Claim Checked Against Source Data

Five questions were tested after both fixes — one for each data type (sales, inventory, labor, customer), plus one narrow single-store lookup to confirm the original semantic path still works correctly.

Each subsection below reproduces the source-data check used to validate the bot's actual claims.


### 6.1 Revenue Underperformance (sales)
**Question:** *"Which stores are underperforming against revenue targets?"*


In [ ]:
w12_sales = sales[sales['week'] == '2024-W12'].sort_values('revenue_variance_pct')
underperformers = w12_sales[w12_sales['revenue_variance_pct'] < 0]
print(f"Stores below target in 2024-W12: {len(underperformers)}")
print(underperformers[['store_id', 'revenue_actual', 'revenue_target', 'revenue_variance_pct', 'same_store_sales_growth']].to_string(index=False))


**Result:** the bot's answer cited all 9 underperforming stores, in the correct order, with every revenue figure, target, and variance percentage matching source data exactly. ✅


### 6.2 Shrinkage Rates (inventory)
**Question:** *"Which stores have the worst shrinkage rates?"*


In [ ]:
w12_inv = inventory[inventory['week'] == '2024-W12'].sort_values('shrinkage_rate', ascending=False)
print("Top 8 stores by shrinkage rate, 2024-W12:")
print(w12_inv[['store_id', 'shrinkage_rate', 'stockout_incidents', 'inventory_turnover']].head(8).to_string(index=False))


**Result:** Miami (4.34%) and Philadelphia (2.88%) correctly identified as the only stores breaching the 2% benchmark; the six "near-threshold" stores listed matched exactly, in the correct order, along with supporting stockout and turnover figures. ✅


### 6.3 Labor Cost Overruns (labor)
**Question:** *"Where are labor costs running over budget?"* (post Bug #2 fix)


In [ ]:
w12_labor = labor[labor['week'] == '2024-W12'].copy()
w12_labor['over_dollar'] = w12_labor['labor_cost_actual'] - w12_labor['labor_cost_target']
w12_labor = w12_labor.sort_values('labor_variance_pct', ascending=False)

print("Top 5 critical overruns, 2024-W12:")
print(w12_labor[['store_id', 'labor_variance_pct', 'over_dollar', 'hours_worked', 'hours_scheduled']].head(5).to_string(index=False))

print()
print("Stores at or under budget:")
under_budget = w12_labor[w12_labor['labor_variance_pct'] < 1.0].sort_values('labor_variance_pct')
print(under_budget[['store_id', 'labor_variance_pct']].to_string(index=False))


**Result:** the top 5 critical overruns (Chicago, New York, Seattle, Boston, Baltimore) all matched source data exactly on percentage, dollar overage, and hours.

**One discrepancy found:** Nashville (S024) was listed by the bot as "-0.7% under budget" in its summary table. The source data shows **+0.7%** — a slight overage, not an underage. Minor in magnitude, but a real sign error, caught only because every number was checked against ground truth rather than accepted at face value. ⚠️


### 6.4 NPS Scores (customer)
**Question:** *"Which stores have the lowest NPS scores?"*


In [ ]:
w12_cust = customer[customer['week'] == '2024-W12'].sort_values('nps_score')
print("Bottom 5 stores by NPS, 2024-W12:")
print(w12_cust[['store_id', 'nps_score', 'complaints_logged', 'return_rate', 'loyalty_members_active']].head(5).to_string(index=False))


**Result:** all 5 lowest-NPS stores and their supporting complaint/return-rate/loyalty figures matched source data exactly, in the correct order. ✅


### 6.5 Single-Store Lookup — Narrow Path (sales)
**Question:** *"How did store S016 do last week?"*

This question intentionally targets the **original semantic retrieval path** (unchanged by either fix) to confirm it still works correctly for specific, single-entity questions.


In [ ]:
s016 = sales[sales['store_id'] == 'S016'].sort_values('week')
print("S016 (New York) — recent weeks:")
print(s016[['week', 'revenue_actual', 'revenue_target', 'revenue_variance_pct', 'conversion_rate', 'foot_traffic']].tail(5).to_string(index=False))


**Result:** the bot correctly identified W08 as the most recent available week (no later weeks existed in the vector store for this store at query time) and all figures — revenue, target, variance, conversion rate, foot traffic — matched source data exactly. ✅


## 7. Takeaways

**Summary across all 5 validation tests:**

| # | Question | Data type | Result |
|---|---|---|---|
| 1 | Revenue underperformance | sales | ✅ Fully correct |
| 2 | Shrinkage rates | inventory | ✅ Fully correct |
| 3 | Labor cost overruns | labor | ✅ Correct, 1 minor sign error found |
| 4 | NPS scores | customer | ✅ Fully correct |
| 5 | Single-store lookup | sales (narrow) | ✅ Fully correct |

**What this process actually demonstrated:**

1. **A RAG system can sound completely confident and still be wrong.** The original answer to the revenue question read fluently and cited real numbers — it just left out most of the dataset, and got two specific claims wrong as a result.

2. **Fixing one bug doesn't mean the underlying fragility is gone.** The same root cause (retrieval not covering the full dataset) resurfaced in a different form (a question phrasing the classifier didn't recognize) immediately after the first fix.

3. **Validation isn't a one-time gate.** Even after both retrieval bugs were fixed, a second pass of validation caught a small sign error (Nashville) in an otherwise fully correct answer. The only way to know that was checking every number against the raw source data — not just checking whether the answer "sounded right."

4. **Keyword-based classification is fast and free, but has a ceiling.** It works well in practice once the pattern list covers common phrasings, but it's fundamentally limited compared to an approach that understands intent rather than matching surface language. Documented here as a known tradeoff, not hidden.

The full pipeline code is available at the project repository, including `query.py` (retrieval + classification logic), `ingest.py` (chunking + embedding), `slack_bot.py` (Slack interface), and `generate_data.py` (synthetic data generation).
